- clean the data 
- drop records containing null timestamp, latitude , longitude
- drop duplicate ids

In [0]:
from pyspark.sql.functions import (
    col, current_timestamp, coalesce, lit, 
    upper, trim, when, count, isnan
)

In [0]:
def quality_report(df, name):
    print(f"\n{'='*45}")
    print(f"  Quality report: {name}")
    print(f"  Total rows: {df.count()}")
    print(f"{'='*45}")
    for c in df.columns:
        null_count = df.filter(col(c).isNull()).count()
        if null_count > 0:
            print(f" {c}: {null_count} nulls")
    print(f"{'='*45}\n")


#### 1. Cleaning vehicle data



In [0]:
# ── 1. Vehicle Data Cleaning ───────────────────────────────────────────────

df_vehicle_bronze = spark.read.table("smart_city.bronze.vehicle_raw")

# Quality check before
quality_report(df_vehicle_bronze, "vehicle BEFORE")

df_vehicle_silver = (df_vehicle_bronze
    .dropna(how="all")
    .dropna(subset=["id", "deviceId", "timestamp",
                    "latitude", "longitude"])
    .dropDuplicates(["id"])
    .withColumn("brand",    trim(col("brand")))
    .withColumn("model",    trim(col("model")))
    .withColumn("fuelType", trim(upper(col("fuelType"))))
    .withColumn("area",     trim(col("area")))
    .withColumn("direction",trim(col("direction")))
    .withColumn("brand",    coalesce(col("brand"),    lit("Unknown")))
    .withColumn("model",    coalesce(col("model"),    lit("Unknown")))
    .withColumn("fuelType", coalesce(col("fuelType"), lit("Unknown")))
    .withColumn("year",     coalesce(col("year"),     lit(0)))
    .withColumn("area",     coalesce(col("area"),     lit("Unknown")))
    .filter(col("speed")     >= 0)
    .filter(col("speed")     <= 300)           # no vehicle goes above 300 km/h
    .filter(col("latitude")  .between(-90, 90))
    .filter(col("longitude") .between(-180, 180))
    .filter(col("year")      .between(1990, 2030))
    .withColumn("processed_at", current_timestamp())
)

# Quality check after
quality_report(df_vehicle_silver, "vehicle AFTER")

df_vehicle_silver.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("smart_city.silver.vehicle_clean")

print("Vehicle silver done")


  Quality report: vehicle BEFORE
  Total rows: 549
 brand: 66 nulls


  Quality report: vehicle AFTER
  Total rows: 425

Vehicle silver done


#### 2. Cleaning GPS data

In [0]:
# ── 2. GPS Data Cleaning ───────────────────────────────────────────────

df_gps_bronze = spark.read.table("smart_city.bronze.gps_raw")

quality_report(df_gps_bronze, "GPS BEFORE")

df_gps_silver = (df_gps_bronze
    .dropna(how="all")
    .dropna(subset=["id", "deviceId", "timestamp",
                    "latitude", "longitude"])
    .dropDuplicates(["id"])
    .withColumn("area",        trim(col("area")))
    .withColumn("direction",   trim(col("direction")))
    .withColumn("vehicleType", trim(col("vehicleType")))
    .withColumn("area",        coalesce(col("area"),        lit("Unknown")))
    .withColumn("direction",   coalesce(col("direction"),   lit("Unknown")))
    .withColumn("vehicleType", coalesce(col("vehicleType"), lit("Unknown")))
    .filter(col("speed")     >= 0)
    .filter(col("speed")     <= 300)
    .filter(col("latitude")  .between(-90, 90))
    .filter(col("longitude") .between(-180, 180))
    .withColumn("processed_at", current_timestamp())
)

quality_report(df_gps_silver, "GPS AFTER")

df_gps_silver.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("smart_city.silver.gps_clean")

print("GPS silver done")


  Quality report: GPS BEFORE
  Total rows: 493


  Quality report: GPS AFTER
  Total rows: 378

GPS silver done


#### 3. Cleaning traffic data

In [0]:
# ── 3. traffic Data Cleaning ───────────────────────────────────────────────

df_traffic_bronze = spark.read.table("smart_city.bronze.traffic_raw")

quality_report(df_traffic_bronze, "traffic BEFORE")

df_traffic_silver = (df_traffic_bronze
    .dropna(how="all")
    .dropna(subset=["id", "deviceId", "cameraId", "timestamp"])
    .dropDuplicates(["id"])
    .withColumn("area",     trim(col("area")))
    .withColumn("cameraId", trim(col("cameraId")))
    .withColumn("area",     coalesce(col("area"),     lit("Unknown")))
    .withColumn("snapshot", coalesce(col("snapshot"), lit("No snapshot")))
    .withColumn("has_snapshot",
        when(col("snapshot") != "No snapshot", True).otherwise(False))
    .filter(col("latitude")  .between(-90, 90))
    .filter(col("longitude") .between(-180, 180))
    .withColumn("processed_at", current_timestamp())
)

quality_report(df_traffic_silver, "traffic AFTER")

df_traffic_silver.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("smart_city.silver.traffic_clean")

print("Traffic silver done")


  Quality report: traffic BEFORE
  Total rows: 444


  Quality report: traffic AFTER
  Total rows: 354

Traffic silver done


#### 4. Cleaning Weather data

In [0]:
# ── 4. weather Data Cleaning ───────────────────────────────────────────────

df_weather_bronze = spark.read.table("smart_city.bronze.weather_raw")

quality_report(df_weather_bronze, "weather BEFORE")

df_weather_silver = (df_weather_bronze
    .dropna(how="all")
    .dropna(subset=["id", "deviceId", "timestamp", "area"])
    .dropDuplicates(["id"])
    .withColumn("area",              trim(col("area")))
    .withColumn("weather_condition", trim(col("weather_condition")))
    .withColumn("area",              coalesce(col("area"),
                                              lit("Unknown")))
    .withColumn("weather_condition", coalesce(col("weather_condition"),
                                              lit("Unknown")))
    .filter(col("temperature") .between(-50, 60))  # realistic earth temp range
    .filter(col("humidity")    .between(0, 100))
    .filter(col("wind_speed")  >= 0)
    .filter(col("wind_speed")  <= 400)             # max recorded wind speed
    .withColumn("processed_at", current_timestamp())
)

quality_report(df_weather_silver, "weather AFTER")

df_weather_silver.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("smart_city.silver.weather_clean")

print("Weather silver done")


  Quality report: weather BEFORE
  Total rows: 229


  Quality report: weather AFTER
  Total rows: 203

Weather silver done


#### 5. Cleaning Emergency data

In [0]:
# ── 5. Emergency Data Cleaning ───────────────────────────────────────────────

df_emergency_bronze = spark.read.table("smart_city.bronze.emergency_raw")

quality_report(df_emergency_bronze, "emergency BEFORE")

# Valid status values we expect
valid_statuses = ["ACTIVE", "RESOLVED", "PENDING", "CANCELLED"]

df_emergency_silver = (df_emergency_bronze
    .dropna(how="all")
    .dropna(subset=["id", "incidentId", "deviceId", "timestamp"])
    .dropDuplicates(["id"])
    .withColumn("status", trim(upper(col("status"))))
    .withColumn("area",   trim(col("area")))
    .withColumn("area",   coalesce(col("area"),   lit("Unknown")))
    .withColumn("status", coalesce(col("status"), lit("UNKNOWN")))
    .withColumn("status",
        when(col("status").isin(valid_statuses), col("status"))
        .otherwise(lit("UNKNOWN")))
    .filter(col("latitude")  .between(-90, 90))
    .filter(col("longitude") .between(-180, 180))
    .withColumn("processed_at", current_timestamp())
)

quality_report(df_emergency_silver, "emergency AFTER")

df_emergency_silver.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("smart_city.silver.emergency_clean")

print(" Emergency silver done")


  Quality report: emergency BEFORE
  Total rows: 511


  Quality report: emergency AFTER
  Total rows: 401

 Emergency silver done


#### Join (GPS + Vehicle)

In [0]:
df_vehicle_ref = (spark.read.table("smart_city.silver.vehicle_clean")
    .select("deviceId", "brand", "model", "year", "fuelType")
    .dropDuplicates(["deviceId"]))

df_gps_clean = spark.read.table("smart_city.silver.gps_clean")

df_journey = (df_gps_clean
    .join(df_vehicle_ref, on="deviceId", how="left")
    .withColumn("brand",    coalesce(col("brand"),    lit("Unknown")))
    .withColumn("model",    coalesce(col("model"),    lit("Unknown")))
    .withColumn("fuelType", coalesce(col("fuelType"), lit("Unknown")))
    .withColumn("year",     coalesce(col("year"),     lit(0)))
)

unmatched = df_journey.filter(col("brand") == "Unknown").count()
print(f"GPS rows with no vehicle match: {unmatched}")

df_journey.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("smart_city.silver.vehicle_journey")

print("Journey table done")

GPS rows with no vehicle match: 238
Journey table done


#### Final Silver Layer Verification

In [0]:
silver_tables = [
    "vehicle_clean",
    "gps_clean",
    "traffic_clean",
    "weather_clean",
    "emergency_clean",
    "vehicle_journey"
]

print("\nSilver layer summary")
print("="*45)
for t in silver_tables:
    count = spark.read.table(f"smart_city.silver.{t}").count()
    print(f"silver.{t:25s} → {count:>6} rows")
print("="*45)


Silver layer summary
silver.vehicle_clean             →    425 rows
silver.gps_clean                 →    378 rows
silver.traffic_clean             →    354 rows
silver.weather_clean             →    203 rows
silver.emergency_clean           →    401 rows
silver.vehicle_journey           →    378 rows
